# ZenteiQ AI Tech Innovations — Task 3: MoE model (DeepSeek)

This notebook provides a detailed setup and execution workflow for training the **DeepSeek-16B-MOE** base model on a **TPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

In [1]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Install MaxText TPU dependencies
!uv pip install -e .[tpu]

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97204, done.
remote: Counting objects: 100% (2279/2279), done.
remote: Compressing objects: 100% (1170/1170), done.
remote: Total 97204 (delta 1841), reused 1144 (delta 1109), pack-reused 94925 (from 5)
Receiving objects: 100% (97204/97204), 411.73 MiB | 1.29 MiB/s, done.
Resolving deltas: 100% (70885/70885), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 99.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 235 packages in 2.24s                                       
Prepared 116 packages in 29.85s                                          
Uninstalled 19 packages in 1.37s
Installed 118 packages in 387ms                             
 - absl-py==1.4.0
 + absl-py==2.4.0
 + antlr4-python3-runtime==4.9.3
 + aqtp==0.9.0
 + astroid==4.0.4
 + astunparse==1.6.3
 + auditwheel==6.7.0
 + black==25.12.0
 + build==1.5.0
 + cfgv==3.5.0
 - chex==0.1.90
 + chex==0.1.92
 + cloud-acceler

In [2]:
!uv pip install qwix tokamax -q

In [3]:
!uv pip install aqtp -q

### 2. Verify JAX TPU Backend Connection

Run the following cell to confirm that JAX successfully detects the TPU backend.

In [2]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX Version: 0.10.2
Available devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Default backend: tpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `deepseek (scaled-down MoE)` | Specifies the base DeepSeek Mixture-of-Experts (MoE) architecture to use. | Loads the DeepSeek model configuration and initializes the transformer with MoE-specific components such as expert layers and routing mechanisms. |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on TPU memory. This allows pure computational throughput benchmarking. |
| **`base_output_directory`** | `./output-deepseek-tpu` | Sets the file storage root path. | All metrics, TensorBoard execution logs, compile metadata, and model checkpoints are saved under this root directory. |
| **`run_name`** | `deepseek-tpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `deepseek-tpu/` for output checkpoints and statistics. |

### 4. Execute GPU Training (Deepseel MOE 16B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [3]:
# !sudo apt-get update && sudo apt-get install -y \
#     libxcomposite1 \
#     libxcursor1 \
#     libxdamage1 \
#     libxext6 \
#     libxfixes3 \
#     libxi6 \
#     libxrandr2 \
#     libxrender1 \
#     libxtst6 \
#     libgbm1 \
#     libasound2



In [4]:
!uv pip uninstall torch -y

Using Python 3.12.13 environment at: /usr
Uninstalled 1 package in 1.36s
 - torch==2.9.0+cpu (from https://download.pytorch.org/whl/cpu/torch-2.9.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl)


In [1]:
# Run training for 50 steps on GPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    hardware=tpu \
    model_name=deepseek2-16b\
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/tpu-output-deepseek2-16b \
    run_name=deepseek2-16b-tpu\
    megablox=false \
    override_model_config=true \
    skip_jax_distributed_system=true \
    base_emb_dim=1152 \
    base_mlp_dim=5120 \
    base_moe_mlp_dim=768 \
    base_num_decoder_layers=12 \
    num_experts=16 \
    num_experts_per_tok=2 \
    shared_experts=1 \
    per_device_batch_size=1 \
    attention=autoselected \
    max_target_length=1024 \
    weight_dtype=bfloat16 \
    grad_dtype=bfloat16 \
    enable_checkpointing=false

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
2026-06-18 18:19:17.260072: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-18 18:19:17.299230: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-18 18:19:18.412441: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[tra

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!cp -r tpu-output-deepseek2-16b /content/drive/MyDrive/
